Here we define a reranker. The idea is to consider the original query from the user and the documents extraced by retrieval using the optimized subqueries. In this case the document we extract will be the metadata of different datasets. Then we provide the result and the original query to a powerful LLM.
In order to do so in the offline phase we generate the instructions from a dataset and then we store them in a vector DB, they have to be linked to the metadata of the dataset they represent.
In the online phase we generate the subqueries and perform the retrieval matching the embeddings in the db. In this way we extract the metadata of multiple datasets. 
Only at this point we can use the reranker.

Partiamo facendo il setup di Chroma

In [2]:
import pandas as pd
from ydata_profiling import ProfileReport
import datamart_profiler
import openai
from dotenv import load_dotenv
from openai import OpenAI
import os
import glob
import json
import sys
sys.path.append(r'c:\Users\ricca\Documents\Polimi\tesi\code')
import utils
import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm  # For progress bar
from multiprocessing import Pool

import importlib
importlib.reload(utils)

import chromadb

In [3]:
try:
    chroma_client = chromadb.PersistentClient()
    chroma_client.get_settings().allow_reset = True
    #chroma_client.reset()
except ValueError as e:
    print(f"Resetting existing Chroma instance: {e}")
    #chroma_client.reset()

embedding_function = utils.OpenAIEmbeddingFunction()

collection = chroma_client.get_or_create_collection(
    name="instructions",
    embedding_function=embedding_function
)

In [7]:
def query_to_vector_db(query_text, n=3):
    results = collection.query(
        query_texts=[query_text],
        n_results=n
    )

    for i, doc in enumerate(results['documents'][0]):
        print(f"{i+1}. {doc} (ID: {results['ids'][0][i]}, Distance: {results['distances'][0][i]:.4f})")

    return results

In [51]:
query = "I need data which contains information about the relations about pregnancy and diabetes"

results = query_to_vector_db(query, 30)

1. Locate a dataset that includes the number of pregnancies and diabetes outcomes. (ID: instruction_3_csv_files\diabetes.csv, Distance: 0.5857)
2. Search for a dataset with patient information including number of pregnancies and heredity factors related to gestational diabetes. (ID: instruction_3_csv_files\Gestational Diabetes.csv, Distance: 0.5992)
3. Find a healthcare dataset related to gestational diabetes. (ID: instruction_0_csv_files\Gestational Diabetes.csv, Distance: 0.6840)
4. Find a dataset that combines patient attributes such as Pregnancies, Glucose, and Age to predict diabetes. (ID: instruction_12_csv_files\diabetes.csv, Distance: 0.6880)
5. Show me datasets that offer numeric patient attributes for studying gestational diabetes outcomes. (ID: instruction_13_csv_files\Gestational Diabetes.csv, Distance: 0.7156)
6. Show me a dataset containing numeric data on age, weight, height, and BMI for predicting gestational diabetes. (ID: instruction_5_csv_files\Gestational Diabetes.c

Ora, dopo aver estratto i risultati migliori abbiamo bisogno che un reranker analizzi il tutto e rimetta in ordine i risultati

In [52]:
unique_results = []
seen = set()

for result in results['metadatas']:
    for metadata in result:
        #print(metadata.get('dataset'))
        if metadata.get('dataset') not in seen:
            seen.add(metadata.get('dataset'))

            metadata_obj = {
            'dataset_name': metadata.get('dataset'),
            'semantic_profile': metadata.get('semantic_profile'),
            'data_profile': metadata.get('data_profile')
        }
            unique_results.append(metadata_obj)

print(f"Unique results: {len(unique_results)}")


Unique results: 7


In [53]:
for metadata in unique_results:
    print(metadata)
    

{'dataset_name': 'csv_files\\diabetes.csv', 'semantic_profile': '{\n  "Pregnancies": {\n    "Temporal": {\n      "isTemporal": false,\n      "resolution": ""\n    },\n    "Spatial": {\n      "isSpatial": false,\n      "resolution": ""\n    },\n    "Entity Type": "Patient Attribute",\n    "Data Format": "Integer",\n    "Domain-Specific Types": "Healthcare",\n    "Function/Usage Context": "Predictor"\n  },\n  "Glucose": {\n    "Temporal": {\n      "isTemporal": false,\n      "resolution": ""\n    },\n    "Spatial": {\n      "isSpatial": false,\n      "resolution": ""\n    },\n    "Entity Type": "Patient Attribute",\n    "Data Format": "Integer",\n    "Domain-Specific Types": "Healthcare",\n    "Function/Usage Context": "Predictor"\n  },\n  "BloodPressure": {\n    "Temporal": {\n      "isTemporal": false,\n      "resolution": ""\n    },\n    "Spatial": {\n      "isSpatial": false,\n      "resolution": ""\n    },\n    "Entity Type": "Patient Attribute",\n    "Data Format": "Integer",\n    

In [54]:
prompt = f"""
Rank the following datasets based on their relevance to the query: "{query}".
The datasets are:
"""

In [55]:
for metadata in unique_results:
    prompt += f"""
    Dataset: {metadata['dataset_name']}
    --------------------
    Semantic Profile:
    {metadata['semantic_profile']}

    Data Profile:
    {metadata['data_profile']}
    --------------------
    """

In [56]:
reranking = utils.call_openai_api(prompt, "o1-mini")

In [58]:
print(reranking.choices[0].message.content)

Based on your query for datasets containing information about the **relationships between pregnancy and diabetes**, here's a ranked list of the provided datasets from **most** to **least** relevant:

1. **`csv_files\Gestational Diabetes.csv`**
   - **Relevance:** Highly relevant as it specifically focuses on **gestational diabetes**, which is diabetes that develops during pregnancy. 
   - **Key Attributes Related to Pregnancy and Diabetes:**
     - **Pregnancy No:** Number of pregnancies.
     - Other health indicators like **BMI**, **Age**, **Prediction**, etc., which can be used to analyze the relationship between pregnancy factors and diabetes outcomes.

2. **`csv_files\diabetes.csv`**
   - **Relevance:** Directly relevant as it includes the **Pregnancies** attribute alongside various diabetes-related indicators.
   - **Key Attributes Related to Pregnancy and Diabetes:**
     - **Pregnancies:** Number of times a patient has been pregnant.
     - **Outcome:** Indicates the presence o